In [1]:
from google.colab import drive
drive.mount('/content/drive')

# 필요한 경우 opencv 설치 (Colab에는 기본적으로 설치되어 있으나 확인용)
# !pip install opencv-python

Mounted at /content/drive


In [4]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# ==========================================
# [1단계] 데이터 전처리 및 Custom Dataset
# ==========================================
class HitAndRunDataset(Dataset):
    def __init__(self, data_dir, clip_length=30, r_value=1.0, resize=(224, 224)):
        self.data_dir = data_dir
        self.clip_length = clip_length
        self.r_value = r_value
        self.resize = resize

        all_mp4_files = [f.split('.')[0] for f in os.listdir(data_dir) if f.endswith('.mp4')]
        self.file_names = []
        for file_name in all_mp4_files:
            txt_path = os.path.join(self.data_dir, f"{file_name}.txt")
            if not os.path.exists(txt_path):
                print(f"Warning: Annotation file not found for {file_name}. Skipping.")
                continue

            # Check if the annotation file contains an 'A' or 'S' action
            has_action = False
            try:
                with open(txt_path, 'r') as f:
                    for line in f.readlines():
                        parts = line.strip().split(',')
                        if parts and len(parts) >= 2 and parts[0] in ['A', 'S']:
                            has_action = True
                            break
            except Exception as e:
                print(f"Error reading annotation file {txt_path}: {e}. Skipping.")
                continue

            if has_action:
                self.file_names.append(file_name)
            else:
                print(f"Warning: No 'A' or 'S' action found in {txt_path}. Skipping.")

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.file_names)

    def _parse_annotation(self, txt_path):
        bboxes = {}
        action = None
        with open(txt_path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split(',')
                if not parts or len(parts) < 2: continue
                if parts[0] == 'car':
                    bboxes[int(parts[1])] = [int(parts[2]), int(parts[3]), int(parts[4]), int(parts[5])]
                elif parts[0] in ['A', 'S']:
                    action = parts
        return bboxes, action

    def _crop_and_pad(self, frame, bbox, r):
        h, w, _ = frame.shape
        x_min, y_min, x_max, y_max = bbox
        veh_w, veh_h = x_max - x_min, y_max - y_min

        new_w, new_h = int(veh_w * r), int(veh_h * r)
        cx, cy = x_min + veh_w // 2, y_min + veh_h // 2

        new_x_min, new_y_min = cx - new_w // 2, cy - new_h // 2
        new_x_max, new_y_max = cx + new_w // 2, cy + new_h // 2

        pad_left, pad_top = max(0, -new_x_min), max(0, -new_y_min)
        pad_right, pad_bottom = max(0, new_x_max - w), max(0, new_y_max - h)

        valid_x_min, valid_y_min = max(0, new_x_min), max(0, new_y_min)
        valid_x_max, valid_y_max = min(w, new_x_max), min(h, new_y_max)

        cropped_frame = frame[valid_y_min:valid_y_max, valid_x_min:valid_x_max]

        if pad_left > 0 or pad_top > 0 or pad_right > 0 or pad_bottom > 0:
            cropped_frame = np.pad(cropped_frame,
                                   ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
                                   mode='constant', constant_values=0)

        return cv2.resize(cropped_frame, self.resize)

    def __getitem__(self, idx):
        file_name = self.file_names[idx]
        mp4_path, txt_path = os.path.join(self.data_dir, f"{file_name}.mp4"), os.path.join(self.data_dir, f"{file_name}.txt")

        bboxes, action = self._parse_annotation(txt_path)
        # At this point, 'action' is guaranteed not to be None due to filtering in __init__
        class_str, target_id, start_f = action[0], int(action[1]), int(action[2])
        label = 1 if class_str == 'A' else 0
        target_bbox = bboxes[target_id]

        cap = cv2.VideoCapture(mp4_path)
        frames = []
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_f)

        for _ in range(self.clip_length):
            ret, frame = cap.read()
            if not ret: break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(self._crop_and_pad(frame, target_bbox, self.r_value))
        cap.release()

        while len(frames) < self.clip_length:
            frames.append(frames[-1] if frames else np.zeros((self.resize[1], self.resize[0], 3), dtype=np.uint8))

        tensor_frames = [self.transform(f) for f in frames]
        video_tensor = torch.stack(tensor_frames).permute(1, 0, 2, 3)

        return video_tensor, torch.tensor(label, dtype=torch.long)

# ==========================================
# [2단계] I3D 기반 3D-CNN 모델 설계
# ==========================================
class BasicConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, **kwargs):
        super(BasicConv3d, self).__init__()
        self.conv = nn.Conv3d(in_channels, out_channels, bias=False, **kwargs)
        self.bn = nn.BatchNorm3d(out_channels)

    def forward(self, x):
        return F.relu(self.bn(self.conv(x)), inplace=True)

class InceptionModule3D(nn.Module):
    def __init__(self, in_channels, out_1x1, red_3x3, out_3x3, red_3x3_2, out_3x3_2, out_pool):
        super(InceptionModule3D, self).__init__()
        self.branch1 = BasicConv3d(in_channels, out_1x1, kernel_size=1)
        self.branch2 = nn.Sequential(BasicConv3d(in_channels, red_3x3, kernel_size=1), BasicConv3d(red_3x3, out_3x3, kernel_size=3, padding=1))
        self.branch3 = nn.Sequential(BasicConv3d(in_channels, red_3x3_2, kernel_size=1), BasicConv3d(red_3x3_2, out_3x3_2, kernel_size=3, padding=1))
        self.branch4 = nn.Sequential(nn.MaxPool3d(kernel_size=3, stride=1, padding=1), BasicConv3d(in_channels, out_pool, kernel_size=1))

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], dim=1)

class HitAndRun3DCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(HitAndRun3DCNN, self).__init__()
        self.conv1 = BasicConv3d(3, 64, kernel_size=7, stride=2, padding=3)
        self.maxpool1 = nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))
        self.conv2 = BasicConv3d(64, 64, kernel_size=1)
        self.conv3 = BasicConv3d(64, 192, kernel_size=3, padding=1)
        self.maxpool2 = nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))
        self.inception1 = InceptionModule3D(192, 64, 96, 128, 16, 32, 32)
        self.inception2 = InceptionModule3D(256, 128, 128, 192, 32, 96, 64)
        self.maxpool3 = nn.MaxPool3d(kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
        self.inception3 = InceptionModule3D(480, 192, 96, 208, 16, 48, 64)

        self.avg_pool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.head_conv = nn.Conv3d(512, num_classes, kernel_size=(1, 1, 1))
        self._initialize_weights()

    def forward(self, x):
        x = self.maxpool1(self.conv1(x))
        x = self.maxpool2(self.conv3(self.conv2(x)))
        x = self.maxpool3(self.inception2(self.inception1(x)))
        x = self.avg_pool(self.inception3(x))
        logits = self.head_conv(x)
        return logits.view(logits.size(0), -1)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

# ==========================================
# [3단계] 학습 루프 및 Colab 실행
# ==========================================
class EarlyStopping:
    def __init__(self, patience=10, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.early_stop = False
        self.counter = 0

    def __call__(self, val_loss):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.counter = 0

def train_model(data_dir):
    # Colab GPU 환경 강제 인식
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"✅ 사용 중인 디바이스: {device}")
    if device.type != 'cuda':
        print("⚠️ 경고: GPU가 활성화되지 않았습니다. Colab 런타임 유형을 확인하세요.")

    # 데이터 로더 (논문 조건: r=1, clip_length=30, batch_size=15)
    train_dataset = HitAndRunDataset(data_dir=data_dir, clip_length=30, r_value=1.0)

    # num_workers=2, pin_memory=True를 추가하여 GPU 데이터 전송 속도 최적화
    train_loader = DataLoader(train_dataset, batch_size=15, shuffle=True, num_workers=2, pin_memory=True)

    # 하이퍼파라미터 설정
    model = HitAndRun3DCNN(num_classes=2).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.00001)
    early_stopping = EarlyStopping(patience=10)
    num_epochs = 100

    print("🚀 GPU 학습을 시작합니다...")
    model.train()

    for epoch in range(num_epochs):
        running_loss = 0.0

        for batch_idx, (inputs, labels) in enumerate(train_loader):
            # GPU 메모리로 텐서 이동
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.6f}")

        # 실제 환경에서는 검증 Loss를 계산하여 early_stopping(val_loss) 호출

    print("✨ 학습 완료!")
    return model

# Colab 실행 진입점
if __name__ == "__main__":
    # TODO: 본인의 구글 드라이브 내 Kaggle 데이터셋 경로로 수정하세요.
    # 예시: '/content/drive/MyDrive/Capstone/kaggle_dataset'
    DATA_DIR = '/content/drive/MyDrive/capstone-26/sample'

    # 폴더가 존재하는지 확인 후 학습 시작
    if os.path.exists(DATA_DIR):
        # 학습 루프 실행
        print(f"데이터 디렉토리 확인 완료: {DATA_DIR}.")
        trained_model = train_model(data_dir=DATA_DIR)
    else:
        print(f"⚠️ 데이터 폴더를 찾을 수 없습니다: {DATA_DIR}")
        print("Google Drive 경로를 올바르게 수정해 주세요.")

✅ 사용 중인 디바이스: cuda
🚀 GPU 학습을 시작합니다...
Epoch [1/100], Loss: 0.0012
Epoch [2/100], Loss: 0.0001
Epoch [3/100], Loss: 0.0001
Epoch [4/100], Loss: 0.0001
Epoch [5/100], Loss: 0.0001
Epoch [6/100], Loss: 0.0000
Epoch [7/100], Loss: 0.0000
Epoch [8/100], Loss: 0.0000
Epoch [9/100], Loss: 0.0000
Epoch [10/100], Loss: 0.0000
Epoch [11/100], Loss: 0.0000
Epoch [12/100], Loss: 0.0000
Epoch [13/100], Loss: 0.0001
Epoch [14/100], Loss: 0.0000
Epoch [15/100], Loss: 0.0000
Epoch [16/100], Loss: 0.0000
Epoch [17/100], Loss: 0.0000
Epoch [18/100], Loss: 0.0000
Epoch [19/100], Loss: 0.0000
Epoch [20/100], Loss: 0.0000
Epoch [21/100], Loss: 0.0000
Epoch [22/100], Loss: 0.0000
Epoch [23/100], Loss: 0.0000
Epoch [24/100], Loss: 0.0000
Epoch [25/100], Loss: 0.0000
Epoch [26/100], Loss: 0.0000
Epoch [27/100], Loss: 0.0000
Epoch [28/100], Loss: 0.0000
Epoch [29/100], Loss: 0.0000
Epoch [30/100], Loss: 0.0000
Epoch [31/100], Loss: 0.0000
Epoch [32/100], Loss: 0.0000
Epoch [33/100], Loss: 0.0000
Epoch [34/100]

In [ ]:
import cv2
import torch
import numpy as np
import torchvision.transforms as transforms
import torch.nn.functional as F

# ==========================================
# 1. CAM 출력을 위한 특징 맵 추출기 (Hook) - 기존과 동일
# ==========================================
activation = {}
def get_activation(name):
    def hook(model, input, output):
        activation[name] = output.detach()
    return hook

# ==========================================
# 2. [수정됨] 전체 화면 CAM 히트맵 생성 및 비디오 저장 함수
# ==========================================
def predict_and_generate_cam_full(model, video_path, txt_path, r_value=1.0, resize=(224, 224), manual_target_id=0):
    print(f"[{video_path.split('/')[-1]}] 전체 화면 CAM 비디오 생성 준비 중...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    handle = model.inception3.register_forward_hook(get_activation('inception3'))

    # 1. 텍스트 파일 파싱
    bboxes = {}
    target_id = None
    with open(txt_path, 'r') as f:
        for line in f.readlines():
            parts = line.strip().split(',')
            if len(parts) < 2: continue
            if parts[0] == 'car':
                bboxes[int(parts[1])] = [int(parts[2]), int(parts[3]), int(parts[4]), int(parts[5])]
            elif parts[0] in ['A', 'S']:
                target_id = int(parts[1])

    if target_id is None:
        target_id = manual_target_id

    if target_id not in bboxes:
        print("⚠️ 텍스트 파일에서 해당 차량의 좌표를 찾을 수 없습니다.")
        return

    target_bbox = bboxes[target_id]

    # 전처리 함수 (모델 입력을 위해 크롭된 텐서 반환)
    def crop_and_pad(frame, bbox, r):
        h, w, _ = frame.shape
        x_min, y_min, x_max, y_max = bbox
        veh_w, veh_h = x_max - x_min, y_max - y_min

        new_w, new_h = int(veh_w * r), int(veh_h * r)
        cx, cy = x_min + veh_w // 2, y_min + veh_h // 2

        new_x_min, new_y_min = cx - new_w // 2, cy - new_h // 2
        new_x_max, new_y_max = cx + new_w // 2, cy + new_h // 2

        pad_left, pad_top = max(0, -new_x_min), max(0, -new_y_min)
        pad_right, pad_bottom = max(0, new_x_max - w), max(0, new_y_max - h)

        valid_x_min, valid_y_min = max(0, new_x_min), max(0, new_y_min)
        valid_x_max, valid_y_max = min(w, new_x_max), min(h, new_y_max)

        cropped_frame = frame[valid_y_min:valid_y_max, valid_x_min:valid_x_max]

        if pad_left > 0 or pad_top > 0 or pad_right > 0 or pad_bottom > 0:
            cropped_frame = np.pad(cropped_frame,
                                   ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)),
                                   mode='constant', constant_values=0)
        return cv2.resize(cropped_frame, resize)

    # 비디오 프레임 추출
    cap = cv2.VideoCapture(video_path)
    original_full_frames = [] # [수정됨] 화면 합성용으로 '원본 전체 프레임'을 저장
    tensor_frames = []        # 모델 입력용 텐서

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    while True:
        ret, frame = cap.read()
        if not ret: break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # 1) 원본 전체 프레임 저장
        original_full_frames.append(frame_rgb)

        # 2) 모델 입력용으로 크롭된 텐서 저장
        processed_frame = crop_and_pad(frame_rgb, target_bbox, r_value)
        tensor_frames.append(transform(processed_frame))

    cap.release()

    total_frames = len(tensor_frames)
    clip_length = 30

    if total_frames < clip_length:
        print("⚠️ 비디오 길이가 짧아 분석할 수 없습니다.")
        return

    full_video_tensor = torch.stack(tensor_frames).permute(1, 0, 2, 3)

    # ----------------------------------------------------
    # [새로운 로직] 원본 좌표계 역산 및 매핑 계산
    # ----------------------------------------------------
    orig_h, orig_w, _ = original_full_frames[0].shape
    x_min, y_min, x_max, y_max = target_bbox
    veh_w, veh_h = x_max - x_min, y_max - y_min

    # r 값(여기선 1.0)에 의해 확장/축소된 실제 Bounding Box 크기
    new_w, new_h = int(veh_w * r_value), int(veh_h * r_value)
    cx, cy = x_min + veh_w // 2, y_min + veh_h // 2

    new_x_min, new_y_min = cx - new_w // 2, cy - new_h // 2
    new_x_max, new_y_max = cx + new_w // 2, cy + new_h // 2

    # 패딩 및 실제 유효 좌표 (원본 화면 내부 좌표)
    pad_left, pad_top = max(0, -new_x_min), max(0, -new_y_min)
    pad_right, pad_bottom = max(0, new_x_max - orig_w), max(0, new_y_max - orig_h)

    valid_x_min, valid_y_min = max(0, new_x_min), max(0, new_y_min)
    valid_x_max, valid_y_max = min(orig_w, new_x_max), min(orig_h, new_y_max)
    # ----------------------------------------------------

    # 비디오 저장 설정 (원본 해상도 유지)
    out_path = '/content/drive/MyDrive/capstone-26/predict_result/output_cam_full.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_video = cv2.VideoWriter(out_path, fourcc, 30.0, (orig_w, orig_h))

    print("🔍 슬라이딩 윈도우 및 CAM 연산 시작...")

    # 처음 29프레임 출력 (타겟 차량 위치에 기본 초록색 박스만 표시)
    for i in range(clip_length - 1):
        frame_bgr = cv2.cvtColor(original_full_frames[i], cv2.COLOR_RGB2BGR)
        cv2.rectangle(frame_bgr, (valid_x_min, valid_y_min), (valid_x_max, valid_y_max), (0, 255, 0), 2)
        out_video.write(frame_bgr)

    with torch.no_grad():
        for i in range(total_frames - clip_length + 1):
            clip = full_video_tensor[:, i:i+clip_length, :, :].unsqueeze(0).to(device)
            outputs = model(clip)
            pred_class = torch.argmax(outputs, dim=1).item()

            feat_map = activation['inception3'].squeeze(0)
            weight = model.head_conv.weight[pred_class]

            cam = torch.sum(weight * feat_map, dim=0)
            cam = F.relu(cam)
            cam_2d = torch.mean(cam, dim=0).cpu().numpy()

            # 0~255로 스케일링
            cam_2d = cam_2d - np.min(cam_2d)
            cam_2d = cam_2d / (np.max(cam_2d) + 1e-8)
            cam_2d_uint8 = np.uint8(255 * cam_2d)

            # [수정됨] 224x224가 아닌 '차량의 실제 크기(new_w, new_h)'로 복원
            cam_resized_padded = cv2.resize(cam_2d_uint8, (new_w, new_h))

            # 화면 밖으로 나갔던 패딩 영역을 잘라내어 유효한 히트맵 추출
            cam_valid = cam_resized_padded[pad_top : new_h - pad_bottom, pad_left : new_w - pad_right]
            heatmap = cv2.applyColorMap(cam_valid, cv2.COLORMAP_JET)

            # 원본 전체 프레임 불러오기
            final_frame = cv2.cvtColor(original_full_frames[i + clip_length - 1], cv2.COLOR_RGB2BGR).copy()
            roi = final_frame[valid_y_min:valid_y_max, valid_x_min:valid_x_max]

            if pred_class == 1: # 충돌 시
                # 타겟 차량 위치(ROI)에만 히트맵 합성
                blended_roi = cv2.addWeighted(roi, 0.6, heatmap, 0.4, 0)
                final_frame[valid_y_min:valid_y_max, valid_x_min:valid_x_max] = blended_roi

                # 시각적 강렬함을 위해 빨간색 바운딩 박스와 텍스트 표시
                cv2.rectangle(final_frame, (valid_x_min, valid_y_min), (valid_x_max, valid_y_max), (0, 0, 255), 3)
                cv2.putText(final_frame, 'CLASS: A (Collision)', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)
            else: # 비충돌(정상) 시
                cv2.rectangle(final_frame, (valid_x_min, valid_y_min), (valid_x_max, valid_y_max), (0, 255, 0), 2)
                cv2.putText(final_frame, 'CLASS: S (Normal)', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 0, 0), 3)

            out_video.write(final_frame)

    out_video.release()
    handle.remove()
    print(f"✨ 전체 화면 CAM 비디오 생성 완료! 파일명: {out_path}")

# ==========================================
# 3. [실행부]
# ==========================================
TARGET_VIDEO_PATH = '/content/drive/MyDrive/capstone-26/sample/220510_LA_0001.mp4'
TARGET_TXT_PATH = '/content/drive/MyDrive/capstone-26/sample/220510_LA_0001.txt'

# 함수명 변경됨 (predict_and_generate_cam_full)
predict_and_generate_cam_full(trained_model, TARGET_VIDEO_PATH, TARGET_TXT_PATH, r_value=1.0)